In [1]:
%load_ext autoreload
%autoreload 2
import torch
from dotenv import load_dotenv
from accelerate import Accelerator
from constant import *
from ChatGpt4Model import ChatGpt4Model
from TrainStrategy import TrainStrategy
from LlmSatdOutputLabelConverter import LlmSatdOutputLabelConverter


/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
prompt_template = PromptTemplate(
    name="Manually Crafted",
    definition="You are a Code Analysis Expert specialized in detecting Self-Admitted Technical Debt (SATD) in Java test code comments. SATD refers to comments where developers acknowledge that the current test implementation is incomplete, suboptimal, or relies on a compromise that should be addressed in the future. These admissions often appear as markers such as TODO or FIXME, or as notes about unresolved issues, temporary fixes, workarounds, hacks, performance limitations, use of deprecated APIs, unsupported features, poor design choices, skipped tests, or uncertain functionality. However, comments that only describe expected behavior, provide instructions, or reference external issues (e.g., JIRA ID) are not SATD unless there is additional information indicating the need for future improvement.",
    instruction="Think step by step and assign the label of **SATD** or **Not-SATD** for each given test code comment.",
    n_shot_template='Comment: "{{ text }}"',
    n_shot_answer_template="Answer: {{ cot }} The answer is **{{ label }}**.",
    line_m_before=3,
    line_n_after=3)
output_label_converter = LlmSatdOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

# Detect with gpt-4o-mini N-Shots


In [4]:
chat_gpt4o_mini_detection_model = ChatGpt4Model('detect', 'gpt-4o-mini', output_label_converter, True)
chat_gpt4o_mini_detection_model.fit(detect_n_shot_dataset)
chat_gpt4o_mini_detection_model.predict(detect_test_dataset.select(range(1)), DETECT_DATASET_NAME, prompt_template,TrainStrategy.N_SHOT_TOP, 4, verbose=True)


gpt-4o-mini-4-shot
[{'role': 'developer', 'content': 'You are a Code Analysis Expert specialized in detecting Self-Admitted Technical Debt (SATD) in Java test code comments. SATD refers to comments where developers acknowledge that the current test implementation is incomplete, suboptimal, or relies on a compromise that should be addressed in the future. These admissions often appear as markers such as TODO or FIXME, or as notes about unresolved issues, temporary fixes, workarounds, hacks, performance limitations, use of deprecated APIs, unsupported features, poor design choices, skipped tests, or uncertain functionality. However, comments that only describe expected behavior, provide instructions, or reference external issues (e.g., JIRA ID) are not SATD unless there is additional information indicating the need for future improvement.\nThink step by step and assign the label of **SATD** or **Not-SATD** for each given test code comment.'}, {'role': 'user', 'content': 'Comment: "//hack

'../cache/output/snapshot/September 15, 2025, 16:54:17$detect_gpt-4o-mini-gpt-4o-mini-4-shot.csv'

# Detect with gpt-4o N-Shots

In [ ]:
chat_gpt4o_detection_model = ChatGpt4Model('detect', 'gpt-4o', output_label_converter, True)
chat_gpt4o_detection_model.fit(detect_n_shot_dataset)
chat_gpt4o_detection_model.predict(detect_test_dataset, DETECT_DATASET_NAME, DETECTION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 4, verbose=False)
